# The Kearns Parameter

A zirconium fuel-clad specification will tell you the alloy, the wall thickness, the
heat treatment — and one texture number, $f$. Not an ODF, not a pole figure: a single
scalar between 0 and 1, quoted along each of three specimen directions.

That is unusual. Texture is a function on a three-dimensional orientation space, and
compressing it to one number normally throws almost everything away. The Kearns
parameter is the case where it does not, and this notebook is about why, how to measure
it four different ways, and how much each way is wrong.

**What you will get out of it**

1. Why one number can stand in for a whole orientation distribution — and the precise
   class of properties for which that is exact rather than approximate.
2. The tensor behind $f$, from which two facts usually quoted as empirical rules follow
   as identities: the three principal values sum to exactly 1, and a random texture
   gives exactly $1/3$.
3. All four measurement routes, cross-checked against simulated textures whose answers
   are known in closed form.
4. Where each route's error comes from, measured rather than asserted: pole-figure
   truncation, ODF kernel smoothing, and the interpolation the diffractogram route needs.
5. The same routes run on real X-ray data from three alloys, where the checks stop
   passing and the reasons become the point.

The derivation is in the theory note
[The Kearns Parameter And Basal-Pole Texture](../../theory/kearns_parameter_and_basal_pole_texture.md);
this notebook is the executable companion.

## 1. Why $f$ exists

A hexagonal crystal is transversely isotropic about $[0001]$. So any second-rank
property of it — thermal expansion, irradiation growth, the second-rank part of elastic
and creep response — is fixed by two numbers, its values along and across the $c$ axis,
and along a direction at angle $\phi$ to $[0001]$ it takes the value

$$P(\phi) = P_{\parallel}\cos^{2}\phi + P_{\perp}\left(1 - \cos^{2}\phi\right).$$

Average that over a polycrystal, weighting each crystal by its volume fraction $V_i$:

$$P_{\mathrm{ref}} = P_{\parallel}\underbrace{\sum_i V_i\cos^{2}\phi_i}_{f}
  + P_{\perp}\left(1 - \sum_i V_i \cos^{2}\phi_i\right)
  = f\,P_{\parallel} + (1-f)\,P_{\perp}.$$

The whole orientation distribution has collapsed into the summation. So

$$\boxed{\;f = \sum_i V_i \cos^{2}\phi_i = \left\langle \cos^{2}\phi \right\rangle\;}$$

is not merely *a* texture index. For this class of properties it is the **exact and
complete** texture input, which is why it outlived the many other scalar measures
proposed alongside it. And it reads physically: the aggregate behaves exactly as if a
fraction $f$ of it were single crystal with $c$ along the reference direction, and the
rest single crystal with $c$ perpendicular to it. Hence *the effective fraction of basal
poles aligned with the direction of interest*.

```{warning}
The reduction holds **only** for properties of that form. Yield strength, fracture
toughness and hydride reorientation are not second-rank properties of this kind;
correlations of those with $f$ are empirical, not derived.
```

## 2. The tensor behind it

Write $\mathbf{c}$ for a crystal's basal pole in the specimen frame and $\mathbf{d}$ for
a specimen direction. Then $\cos\phi = \mathbf{d}\cdot\mathbf{c}$, and the definition is
a quadratic form:

$$f(\mathbf{d}) = \left\langle (\mathbf{d}\cdot\mathbf{c})^{2}\right\rangle
= \mathbf{d}^{\mathsf{T}}\mathbf{A}\,\mathbf{d},
\qquad
\mathbf{A} = \left\langle \mathbf{c}\,\mathbf{c}^{\mathsf{T}}\right\rangle .$$

$\mathbf{A}$ is the second-moment (orientation) tensor of the basal-pole distribution.
Three consequences, and they are the reason this notebook works with $\mathbf{A}$ rather
than with three separately integrated numbers:

- **The sum rule is an identity.** For any orthonormal triad,
  $f_1+f_2+f_3 = \operatorname{tr}\mathbf{A} = \langle\mathbf{c}\cdot\mathbf{c}\rangle = 1$,
  because every $\mathbf{c}$ is a unit vector. Exactly 1, for every texture. So a
  measured triad that misses 1 is reporting **the systematic error of the measurement**,
  not a property of the material — and that is a check needing no reference specimen.
- **Random is $1/3$.** A random texture gives $\mathbf{A}=\mathbf{I}/3$.
- **Every direction, not only three.** One tensor answers for all $\mathbf{d}$, and its
  eigenvalues bound what $f$ can be in any direction. This matters for tubing, whose
  basal maxima sit tens of degrees off the axes $f$ is usually quoted along.

### Setting up: $\alpha$-zirconium

Everything in sections 3–7 is built from simulated textures, so that every number has a
*known* correct answer to check against. Real data arrives in section 8.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.transform import Rotation as SciRot

from pytex import (
    ODF,
    FrameDomain,
    Handedness,
    KernelSpec,
    Lattice,
    OrientationSet,
    Phase,
    PoleFigure,
    ReferenceFrame,
    SymmetrySpec,
    plot_pole_figure,
)
from pytex.core.lattice import CrystalPlane
from pytex.texture.kearns import (
    KEARNS_ISOTROPIC_VALUE,
    kearns_from_odf,
    kearns_from_orientations,
    kearns_from_pole_figure,
    kearns_from_tilt_profile,
    kernel_axis_shrinkage,
    pole_orientation_tensor,
)

crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT)
specimen = ReferenceFrame(
    "sample_rd_td_nd", FrameDomain.SPECIMEN, ("RD", "TD", "ND"), Handedness.RIGHT
)
symmetry = SymmetrySpec.from_point_group("6/mmm", reference_frame=crystal)

# alpha-Zr, c/a = 1.5926. The axial ratio matters: it sets the tilt of every
# (hkil) plane normal to [0001], which is what the diffractogram route reads.
zirconium = Phase(
    "alpha_zr",
    lattice=Lattice(3.232, 3.232, 5.147, 90.0, 90.0, 120.0, crystal_frame=crystal),
    symmetry=symmetry,
    crystal_frame=crystal,
)
basal = CrystalPlane.from_miller_bravais((0, 0, 0, 2), phase=zirconium)

print(f"c/a = {zirconium.lattice.c / zirconium.lattice.a:.4f}")
print(f"random-texture Kearns value = {KEARNS_ISOTROPIC_VALUE:.6f}")

Two helpers build the simulated textures. `orientations_from_c_axes` turns a set of
desired basal-pole directions into orientations, randomizing the one remaining degree of
freedom — rotation about $c$ — so that the result is a *fibre* rather than a component.

`basal_fibre` draws the tilt angles from a density proportional to
$\exp(-\phi^2/2\sigma^2)\,\sin\phi$. The $\sin\phi$ matters: drawing $\phi$ from the
Gaussian directly would give a pole *density on the sphere* diverging as $1/\sin\phi$ at
the fibre axis, which is a singular texture rather than the sharp-but-finite one
intended, and no diffraction method could be judged fairly against it.

In [ ]:
def orientations_from_c_axes(c_axes, *, seed=0):
    """Orientations whose crystal [0001] maps to the given specimen directions."""
    rng = np.random.default_rng(seed)
    axes = np.asarray(c_axes, dtype=float)
    axes = axes / np.linalg.norm(axes, axis=1, keepdims=True)
    helper = np.tile(np.array([1.0, 0.0, 0.0]), (axes.shape[0], 1))
    helper[np.abs(axes[:, 0]) > 0.9] = np.array([0.0, 1.0, 0.0])
    first = np.cross(helper, axes)
    first /= np.linalg.norm(first, axis=1, keepdims=True)
    second = np.cross(axes, first)
    spin = rng.uniform(0.0, 2.0 * np.pi, size=axes.shape[0])
    x_axis = first * np.cos(spin)[:, None] + second * np.sin(spin)[:, None]
    return OrientationSet.from_matrices(
        np.stack([x_axis, np.cross(axes, x_axis), axes], axis=2),
        crystal_frame=crystal,
        specimen_frame=specimen,
        phase=zirconium,
        symmetry=symmetry,
    )


def basal_fibre(axis, *, spread_deg, count=8000, seed=0):
    """c axes about `axis` with a Gaussian pole density of the given halfwidth."""
    rng = np.random.default_rng(seed)
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)
    grid = np.linspace(0.0, np.pi / 2, 4001)
    density = np.exp(-0.5 * (grid / np.deg2rad(spread_deg)) ** 2) * np.sin(grid)
    cumulative = np.cumsum(density)
    tilt = np.interp(rng.uniform(size=count), cumulative / cumulative[-1], grid)
    azimuth = rng.uniform(0.0, 2.0 * np.pi, size=count)
    helper = np.array([1.0, 0.0, 0.0]) if abs(axis[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    u = np.cross(helper, axis)
    u /= np.linalg.norm(u)
    v = np.cross(axis, u)
    c_axes = np.cos(tilt)[:, None] * axis + np.sin(tilt)[:, None] * (
        np.cos(azimuth)[:, None] * u + np.sin(azimuth)[:, None] * v
    )
    return orientations_from_c_axes(c_axes, seed=seed + 1)


def random_texture(count=20000, seed=3):
    return OrientationSet.from_matrices(
        SciRot.random(count, random_state=seed).as_matrix(),
        crystal_frame=crystal,
        specimen_frame=specimen,
        phase=zirconium,
        symmetry=symmetry,
    )

## 3. Three textures, three answers

The three cases the exercise calls for: a random texture, a sharp unimodal basal fibre,
and the bimodal split-basal texture that rolling and pilgering actually produce in
zirconium — two basal maxima tilted symmetrically away from the sheet normal towards TD.
Each has a value of $f$ that can be predicted before running anything, which is what
makes them a check on the code rather than a demonstration of it.

In [ ]:
split_angle_deg = 30.0
azimuth = np.random.default_rng(11).uniform(0.0, 2 * np.pi, 8000)
lobe_sign = np.where(np.random.default_rng(12).uniform(size=8000) < 0.5, 1.0, -1.0)

# Bimodal: two basal maxima at +/- 30 degrees from ND towards TD, each with a
# 12 degree spread. This is the classic rolled/pilgered zirconium texture.
tilt = np.deg2rad(split_angle_deg) + np.deg2rad(12.0) * np.random.default_rng(13).normal(
    size=8000
)
c_axes = np.stack(
    [
        np.zeros_like(tilt),
        lobe_sign * np.sin(tilt),
        np.cos(tilt),
    ],
    axis=1,
)
# scatter each lobe about its own axis so it is a component, not a line
scatter = SciRot.from_rotvec(
    np.deg2rad(8.0) * np.random.default_rng(14).normal(size=(8000, 3))
)
c_axes = scatter.apply(c_axes)

TEXTURES = {
    "random": random_texture(),
    "unimodal basal fibre (10 deg)": basal_fibre([0.0, 0.0, 1.0], spread_deg=10.0, seed=21),
    "bimodal split basal (+/-30 deg to TD)": orientations_from_c_axes(c_axes, seed=15),
    "ideal basal girdle in RD-TD": orientations_from_c_axes(
        np.stack([np.cos(azimuth), np.sin(azimuth), np.zeros_like(azimuth)], axis=1), seed=16
    ),
    "single crystal, c along ND": OrientationSet.from_matrices(
        np.eye(3)[None, :, :],
        crystal_frame=crystal,
        specimen_frame=specimen,
        phase=zirconium,
        symmetry=symmetry,
    ),
}

rows = []
for name, orientations in TEXTURES.items():
    report = kearns_from_orientations(orientations, pole=basal)
    rows.append((name, *np.asarray(report.values), report.triad_sum))

print(f"{'texture':<38} {'f_RD':>7} {'f_TD':>7} {'f_ND':>7} {'sum':>10}")
for name, f_rd, f_td, f_nd, total in rows:
    print(f"{name:<38} {f_rd:7.4f} {f_td:7.4f} {f_nd:7.4f} {total:10.6f}")

Read the table against what the geometry demands:

- **Random** gives $1/3$ everywhere, to sampling error.
- **The unimodal fibre** puts almost everything along ND. It cannot reach 1 unless the
  spread is zero.
- **The bimodal texture** is the interesting one. Its basal poles are nowhere near RD, so
  $f_{\mathrm{RD}}$ is small; splitting towards TD moves weight out of ND and into TD.
  This asymmetry between TD and RD is the whole reason zirconium components are
  anisotropic, and it is exactly what $f$ was invented to quantify.
- **The girdle** gives $(1/2, 1/2, 0)$ exactly — the most extreme texture that still
  keeps $f$ below $1/2$ in every direction.
- **The single crystal** gives $(0,0,1)$.

And every row sums to 1 to twelve decimal places, because it must.

### The tensor itself

`kearns_from_orientations` returns the whole tensor, not only the three axis values, so
any direction can be asked for afterwards — and the eigenvalues say what $f$ *could* be
in the best and worst direction. For the bimodal texture the largest principal value is
noticeably above $f_{\mathrm{ND}}$, because the two lobes' mean axis is not ND.

In [ ]:
bimodal_report = kearns_from_orientations(
    TEXTURES["bimodal split basal (+/-30 deg to TD)"], pole=basal
)
tensor = np.asarray(bimodal_report.orientation_tensor)

print("pole orientation tensor A =")
print(np.array2string(tensor, precision=4, suppress_small=True))
print(f"\ntrace = {np.trace(tensor):.12f}  (the sum rule, as an identity)")
print("principal values:", np.round(np.asarray(bimodal_report.principal_values), 4))

# The same tensor, built directly from the basal poles.
poles = TEXTURES["bimodal split basal (+/-30 deg to TD)"].map_crystal_directions(basal.normal)
direct = pole_orientation_tensor(np.asarray(getattr(poles, "values", poles)))
print("matches pole_orientation_tensor:", bool(np.allclose(direct, tensor, atol=1e-12)))

# f along an arbitrary direction, from the same tensor.
diagonal = kearns_from_orientations(
    TEXTURES["bimodal split basal (+/-30 deg to TD)"],
    pole=basal,
    directions=[[0.0, 1.0, 1.0]],
    direction_labels=("halfway TD-ND",),
)
print(f"\nf halfway between TD and ND = {diagonal.value('halfway TD-ND'):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15.0, 4.2), subplot_kw={"aspect": "equal"})
for ax, (name, orientations) in zip(axes, list(TEXTURES.items())[:4], strict=True):
    figure = PoleFigure.from_orientations(orientations, basal)
    plot_pole_figure(figure, ax=ax, title=name)
fig.suptitle("(0002) pole figures of the simulated textures", y=1.02)
fig.tight_layout()

### $f$ is a smooth function of texture sharpness

Sweeping the fibre spread traces the whole range from single crystal to random, and
shows how insensitive $f$ is at the sharp end: a fibre has to be quite diffuse before
$f_{\mathrm{ND}}$ leaves the 0.8s. That is worth knowing before quoting $f$ to three
decimal places as a process-control variable.

In [ ]:
spreads = np.array([2.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 40.0, 55.0, 75.0, 100.0])
fibre_values = np.array(
    [
        kearns_from_orientations(
            basal_fibre([0.0, 0.0, 1.0], spread_deg=s, count=20000, seed=31), pole=basal
        ).value("ND")
        for s in spreads
    ]
)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(spreads, fibre_values, "o-", label=r"$f_{\rm ND}$ of a basal fibre")
ax.axhline(KEARNS_ISOTROPIC_VALUE, ls="--", color="0.4", label="random, 1/3")
ax.set_xlabel("fibre halfwidth (degrees)")
ax.set_ylabel(r"$f_{\rm ND}$")
ax.set_ylim(0.0, 1.0)
ax.legend()
ax.set_title("Kearns parameter against texture sharpness")
fig.tight_layout()

for s, value in zip(spreads[:6], fibre_values[:6], strict=True):
    print(f"  halfwidth {s:5.1f} deg -> f_ND = {value:.4f}")

## 4. The tilt profile: why azimuth does not matter

Only $\phi$ enters the definition, so the azimuthal detail of a pole figure is
irrelevant to $f$. Kearns' second observation was that this lets the reference direction
be treated as a fibre axis: average the pole density over the full $360^\circ$ of
rotation about it to get a one-dimensional profile $I(\phi)$, and then

$$f = \frac{\int_0^{\pi/2} I(\phi)\,\sin\phi\,\cos^{2}\phi\,\mathrm{d}\phi}
            {\int_0^{\pi/2} I(\phi)\,\sin\phi\,\mathrm{d}\phi}.$$

The $\sin\phi$ converts pole *density* to *volume fraction*: the ring of orientations at
tilt $\phi$ has circumference proportional to $\sin\phi$. Two consequences are routinely
misread off a pole figure, and the plot below makes both visible.

In [ ]:
phi = np.linspace(0.0, np.pi / 2, 400)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(np.degrees(phi), np.sin(phi), label=r"$\sin\phi$  (volume weight)")
ax.plot(np.degrees(phi), np.cos(phi) ** 2, label=r"$\cos^{2}\phi$  (the property factor)")
ax.plot(
    np.degrees(phi),
    np.sin(phi) * np.cos(phi) ** 2,
    lw=2.5,
    label=r"$\sin\phi\,\cos^{2}\phi$  (the integrand)",
)
peak = np.degrees(phi[np.argmax(np.sin(phi) * np.cos(phi) ** 2)])
ax.axvline(peak, ls=":", color="0.4")
ax.set_xlabel(r"tilt $\phi$ from the reference direction (degrees)")
ax.set_ylabel("weight")
ax.legend()
ax.set_title("What each tilt contributes to $f$")
fig.tight_layout()

print(f"The numerator's weight peaks at phi = {peak:.1f} deg, not at 0.")
print("At phi = 0 the volume fraction is zero however intense the pole figure centre is.")

So a bright spot at the centre of a pole figure contributes *nothing* to $f$ — the band
it occupies has no area — while the data between $50^\circ$ and $90^\circ$ of tilt, which
is exactly where a reflection measurement is weakest, carries most of the weight. That
single fact explains most of what goes wrong in section 8.

### Reproducing Kearns' own calculation, column by column

Kearns did not only publish the method; he published a worked instance of it. Table 3 of
WAPD-TM-472 tabulates the arithmetic for a swaged Zircaloy-2 rod on two perpendicular
sections, in five columns:

| column | symbol | what it is |
| --- | --- | --- |
| 1 | $\Delta\phi$ | the tilt bin, ten degrees wide |
| 2 | $I_\phi(\mathrm{av.})$ | measured pole density in the bin, times-random |
| 3 | $I_\phi\sin\bar\phi$ | density converted to volume |
| 4 | $V_{\Delta\phi}$ | volume fraction, column 3 normalised |
| 5 | $V_{\Delta\phi}\cos^{2}\bar\phi$ | that volume's contribution to $f$ |

and $f$ is the sum of column 5. Reproducing it is the strongest available check on an
implementation, because the intermediate columns localise any disagreement to a single
step rather than leaving one number to argue about.

The two blocks behave differently, and both are instructive:

- the **longitudinal section** (reference direction *perpendicular* to the rod axis)
  reproduces to the last printed digit;
- the **transverse section** (reference direction *along* the rod axis) reproduces in
  every column except one cell — and that cell propagates into his quoted $f$.

Both blocks are worked through below one step at a time, with his printed values beside
ours at every stage.

In [ ]:
# Kearns (1965) WAPD-TM-472, Table 3, transcribed exactly as printed.
#
# His transverse block prints a single "0 - 50" row of zeros; it is expanded to
# five zero bins here so both blocks live on the same ten-degree grid. That is a
# change of layout, not of content: a zero density contributes zero to every
# column downstream.
bin_lower = np.arange(0.0, 90.0, 10.0)
bin_upper = bin_lower + 10.0
phi_mid = bin_lower + 5.0            # the midpoints Kearns read his curves at
phi_rad = np.deg2rad(phi_mid)

KEARNS_TABLE_3 = {
    "longitudinal": {
        "I": np.array([3.27, 2.71, 1.69, 1.35, 1.17, 0.97, 0.73, 0.62, 0.55]),
        "I_sin": np.array([0.285, 0.702, 0.715, 0.775, 0.827, 0.794, 0.661, 0.600, 0.548]),
        "V": np.array([0.048, 0.119, 0.121, 0.131, 0.140, 0.134, 0.117, 0.101, 0.093]),
        "V_cos2": np.array(
            [0.0478, 0.111, 0.0993, 0.0879, 0.0700, 0.0441, 0.0209, 0.0061, 0.0007]
        ),
        "printed_I_sin_sum": 5.91,
        "printed_f": 0.488,
    },
    "transverse": {
        "I": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.11, 0.85, 2.43, 3.48]),
        "I_sin": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.090, 0.770, 2.35, 3.47]),
        "V": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.014, 0.116, 0.353, 0.521]),
        "V_cos2": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0046, 0.0208, 0.0214, 0.0040]),
        "printed_I_sin_sum": 6.68,
        "printed_f": 0.0508,
    },
}


def kearns_columns(density):
    """Columns 3, 4 and 5 of Table 3, from column 2."""
    i_sin = density * np.sin(phi_rad)          # step 2: density -> volume
    volume = i_sin / i_sin.sum()               # step 3: normalise to fractions
    contribution = volume * np.cos(phi_rad) ** 2  # step 4: weight by cos^2
    return i_sin, volume, contribution


for name, block in KEARNS_TABLE_3.items():
    print(f"{name:>13} section: {int(np.count_nonzero(block['I']))} non-zero bins, "
          f"printed f = {block['printed_f']}")

#### Step 1 — the measurement: $I_\phi$, an azimuthally averaged pole density

Column 2 is the only input. It is the $(0001)$ pole density averaged over the full
$360^\circ$ of rotation about the reference direction, read off the $I_\phi$ curve at each
bin's midpoint, in multiples of a random distribution. Two things to notice before any
arithmetic happens, because they already determine the answer's shape:

- the **longitudinal** column falls monotonically from $3.27$ at the centre to $0.55$ at
  the rim — basal poles concentrated near the reference direction;
- the **transverse** column is the mirror image: identically zero out to $50^\circ$, then
  rising to $3.48$ at the rim — basal poles concentrated *perpendicular* to the reference
  direction, which is the axial direction of a swaged rod.

So before computing anything we can predict $f$ will be large for the first and near zero
for the second. Note also that column 2 is *not* a volume fraction: reading these numbers
as "how much material" is the mistake step 2 exists to prevent.

In [ ]:
print(f"{'bin':>9} {'phi_mid':>8} {'I_long':>8} {'I_trans':>9}")
for k in range(9):
    print(
        f"{int(bin_lower[k]):3d}-{int(bin_upper[k]):<5d} {phi_mid[k]:7.0f}d "
        f"{KEARNS_TABLE_3['longitudinal']['I'][k]:8.2f} "
        f"{KEARNS_TABLE_3['transverse']['I'][k]:9.2f}"
    )
print()
print("Column 2 is a density, not an amount. The longitudinal column peaks at the")
print("figure centre; the transverse one is zero there. Neither statement is yet a")
print("statement about volume.")

#### Step 2 — $\times \sin\bar\phi$: from density to volume

The crystals whose basal pole is tilted by $\phi$ form a ring on the reference sphere, and
that ring's circumference is proportional to $\sin\phi$. So the *amount of material* in a
bin is its density times $\sin\bar\phi$, not its density:

$$\mathrm{d}V \propto I(\phi)\,\sin\phi\,\mathrm{d}\phi .$$

This is the step that overturns the naive reading of a pole figure. In the longitudinal
block the density falls by a factor of six from centre to rim, but column 3 *rises* — from
$0.285$ to a plateau near $0.8$ — because the widening ring more than compensates. The
brightest bin of the pole figure contributes the least material of any bin.

In [ ]:
print(f"{'bin':>9} {'sin(phi)':>9} | {'LONGITUDINAL':^26} | {'TRANSVERSE':^26}")
print(f"{'':9} {'':9} | {'I':>7} {'I sin':>8} {'Kearns':>8} | {'I':>7} {'I sin':>8} {'Kearns':>8}")
ls, ts = KEARNS_TABLE_3["longitudinal"], KEARNS_TABLE_3["transverse"]
ls_sin, ls_v, ls_c = kearns_columns(ls["I"])
ts_sin, ts_v, ts_c = kearns_columns(ts["I"])
for k in range(9):
    print(
        f"{int(bin_lower[k]):3d}-{int(bin_upper[k]):<5d} {np.sin(phi_rad[k]):9.4f} "
        f"| {ls['I'][k]:7.2f} {ls_sin[k]:8.3f} {ls['I_sin'][k]:8.3f} "
        f"| {ts['I'][k]:7.2f} {ts_sin[k]:8.3f} {ts['I_sin'][k]:8.3f}"
    )
print(f"{'sum':>9} {'':9} | {'':7} {ls_sin.sum():8.3f} {ls['printed_I_sin_sum']:8.2f} "
      f"| {'':7} {ts_sin.sum():8.3f} {ts['printed_I_sin_sum']:8.2f}")
print()
print(f"Longitudinal: density falls {ls['I'][0] / ls['I'][-1]:.1f}x from centre to rim,")
print(f"but the volume column rises {ls_sin[-1] / ls_sin[0]:.1f}x over the same span.")

#### Step 3 — normalise: $V_{\Delta\phi}$, the volume fractions

Column 4 divides column 3 by its own sum, so the fractions add to 1 and describe how the
material is distributed over tilt. This is also the step that makes $f$ **scale-free**:
whatever units column 2 was in — times-random, counts per second, arbitrary — the ratio
removes them. That is why the diffractogram route can use calculated reference intensities
in arbitrary units and still get the right $f$.

His column sums to $1.004$ rather than $1.000$ in both blocks, because he rounded each
entry to three decimals independently. That rounding sets the scale of disagreement we
should tolerate everywhere else in the table — and it is the yardstick against which the
one bad cell in the next step will be judged.

In [ ]:
print(f"{'bin':>9} | {'LONGITUDINAL':^19} | {'TRANSVERSE':^19}")
print(f"{'':9} | {'V':>8} {'Kearns':>8} | {'V':>8} {'Kearns':>8}")
for k in range(9):
    print(
        f"{int(bin_lower[k]):3d}-{int(bin_upper[k]):<5d} "
        f"| {ls_v[k]:8.3f} {ls['V'][k]:8.3f} | {ts_v[k]:8.3f} {ts['V'][k]:8.3f}"
    )
print(f"{'sum':>9} | {ls_v.sum():8.3f} {ls['V'].sum():8.3f} | {ts_v.sum():8.3f} {ts['V'].sum():8.3f}")
print()
print(f"His columns sum to {ls['V'].sum():.3f} and {ts['V'].sum():.3f} rather than 1.000:")
print("three-decimal rounding, applied per row. Disagreements of that size are expected.")
print()
print("Where the material actually is:")
print(f"  longitudinal: {100 * ls_v[:4].sum():.0f} percent of it below 40 degrees of tilt")
print(f"  transverse:   {100 * ts_v[-2:].sum():.0f} percent of it above 70 degrees")

#### Step 4 — $\times \cos^{2}\bar\phi$: each bin's contribution to $f$

Column 5 applies the property weight. $\cos^{2}\phi$ falls from $0.99$ at the first bin to
$0.008$ at the last, so it does the opposite of $\sin\phi$: it suppresses exactly the
high-tilt bins that step 2 promoted. $f$ is the sum of this column, and the two blocks now
separate completely — the longitudinal one keeps most of its volume where $\cos^{2}$ is
large, the transverse one has put all of its volume where $\cos^{2}$ is almost zero.

In [ ]:
print("LONGITUDINAL SECTION")
print(f"{'bin':>9} {'cos2':>8} {'V':>8} {'V cos2':>9} {'Kearns':>9} {'resid':>9}")
for k in range(9):
    print(
        f"{int(bin_lower[k]):3d}-{int(bin_upper[k]):<5d} {np.cos(phi_rad[k])**2:8.4f} "
        f"{ls_v[k]:8.3f} {ls_c[k]:9.4f} {ls['V_cos2'][k]:9.4f} "
        f"{ls_c[k] - ls['V_cos2'][k]:+9.4f}"
    )
print(f"{'f':>9} {'':8} {'':8} {ls_c.sum():9.4f} {ls['printed_f']:9.4f} "
      f"{ls_c.sum() - ls['printed_f']:+9.4f}")
print()
print(f"largest single-cell disagreement: {np.abs(ls_c - ls['V_cos2']).max():.4f}")
print("Every cell, and the total, agree to within his own rounding. The longitudinal")
print("block reproduces.")

The longitudinal block is settled: the total is $0.4879$ against his printed $0.488$, and
no individual cell disagrees by more than his three-decimal rounding can explain. That is
the number this repository pins as a regression baseline.

Now the transverse block, computed exactly the same way.

In [ ]:
print("TRANSVERSE SECTION")
print(f"{'bin':>9} {'cos2':>8} {'V':>8} {'V cos2':>9} {'Kearns':>9} {'resid':>9}")
for k in range(9):
    flag = "  <-- ?" if abs(ts_c[k] - ts["V_cos2"][k]) > 1e-3 else ""
    print(
        f"{int(bin_lower[k]):3d}-{int(bin_upper[k]):<5d} {np.cos(phi_rad[k])**2:8.4f} "
        f"{ts_v[k]:8.3f} {ts_c[k]:9.4f} {ts['V_cos2'][k]:9.4f} "
        f"{ts_c[k] - ts['V_cos2'][k]:+9.4f}{flag}"
    )
print(f"{'f':>9} {'':8} {'':8} {ts_c.sum():9.4f} {ts['printed_f']:9.4f} "
      f"{ts_c.sum() - ts['printed_f']:+9.4f}")

residuals = np.abs(ts_c - ts["V_cos2"])
worst = int(np.argmax(residuals))
others = np.delete(residuals, worst)
print()
print(f"worst cell: bin {int(bin_lower[worst])}-{int(bin_upper[worst])}, "
      f"off by {residuals[worst]:.4f}")
print(f"every other cell in this block: at most {others.max():.4f}")
print(f"ratio: {residuals[worst] / others.max():.0f}x")
print(f"largest disagreement anywhere in the longitudinal block: "
      f"{np.abs(ls_c - ls['V_cos2']).max():.4f}")

#### The one bad cell

The disagreement is not spread across the block — it sits entirely in the
$70$–$80^{\circ}$ row, an order of magnitude larger than any other cell in either block.
That isolation is what distinguishes an arithmetic slip from accumulated rounding: rounding
error is small and everywhere, a slip is large and in one place.

The row's own arithmetic is unambiguous, because every input to it is printed on the same
line:

$$V_{\Delta\phi}\cos^{2}\bar\phi = 0.353 \times \cos^{2}(75^\circ) = 0.353 \times 0.06699 = 0.0237,$$

against the printed $0.0214$. And his quoted total is the sum of the printed column
*including* the bad cell, so the error was made before summing and carried into $f$.

In [ ]:
v70, cos2_75 = ts["V"][7], float(np.cos(np.deg2rad(75.0)) ** 2)
print(f"the 70-80 row, from its own printed inputs:")
print(f"  V = {v70},  cos^2(75 deg) = {cos2_75:.5f}")
print(f"  V cos^2 = {v70 * cos2_75:.4f}      printed: {ts['V_cos2'][7]:.4f}")
print()
print("His printed total is the sum of his printed column, bad cell included:")
print(f"  sum of printed column   = {ts['V_cos2'].sum():.4f}   quoted f = {ts['printed_f']}")
print(f"  same column, cell fixed = {ts['V_cos2'].sum() - ts['V_cos2'][7] + v70 * cos2_75:.4f}")
print()
print("So the corrected transverse f, three ways:")
print(f"  from his volume-fraction column   f = {float((ts['V'] * np.cos(phi_rad) ** 2).sum()):.4f}")
print(f"  from his intensity column         f = {ts_c.sum():.4f}")
print(f"  as printed                        f = {ts['printed_f']:.4f}")
print()
print(f"The quoted value is low by {100 * (1 - ts['printed_f'] / ts_c.sum()):.0f} percent.")

So the transverse Kearns parameter of that rod is about $0.053$, not the printed $0.0508$.
A digit transposition of $0.0241$ would land close to the printed figure, but the table
alone cannot establish the mechanism, and it does not matter much: what matters is that
the row is inconsistent with its own printed inputs, and that the error is confined to
that row.

Three things worth taking from it, none of which is "the method is wrong":

1. **The method is unaffected.** Kearns' text draws the right conclusion from these
   numbers — $f \approx 0.49$ and $\approx 0.05$ on the two sections, "very close to the
   values of 0.5 and 0.0 which would correspond to a perfect fiber texture". A three-percent
   error in the smaller of two numbers changes none of that.
2. **This is why intermediate columns are published.** With only the final $f$ printed, the
   disagreement would be unattributable — a difference in quadrature, in bin midpoints, in
   the reading of a graph. With columns 3, 4 and 5 printed, it localises to one cell in one
   row in one step.
3. **It is why the longitudinal block is the pinned baseline.** A regression test wants a
   reference value that is *right*, not merely published.

In [ ]:
# The library function, on both blocks, agreeing with the hand calculation above.
computed = {}
for name, block in KEARNS_TABLE_3.items():
    report = kearns_from_tilt_profile(
        phi_mid, block["I"], pole=basal, specimen_frame=specimen
    )
    _, _, contribution = kearns_columns(block["I"])
    computed[name] = report.value("ND")
    print(
        f"{name:>13}: kearns_from_tilt_profile -> {report.value('ND'):.6f}   "
        f"columns by hand -> {contribution.sum():.6f}   "
        f"Kearns printed -> {block['printed_f']}"
    )

# Now the sum rule, which these two measurements are enough to test. A swaged rod
# is axially symmetric, so the two directions perpendicular to its axis are
# equivalent and share one value. The longitudinal section measures one of them;
# the transverse section measures the axis. The triad is therefore
# (f_perp, f_perp, f_axial), and it must sum to 1.
f_perp, f_axial = computed["longitudinal"], computed["transverse"]
print()
print(f"  f perpendicular to the rod axis (longitudinal section) = {f_perp:.4f}  [twice]")
print(f"  f along the rod axis            (transverse section)   = {f_axial:.4f}")
print(f"  triad sum = 2 x {f_perp:.4f} + {f_axial:.4f} = {2 * f_perp + f_axial:.4f}")
print(f"  closure error = {100 * abs(2 * f_perp + f_axial - 1.0):.1f} percent")
print()
print("  the same test on his printed values: "
      f"{2 * KEARNS_TABLE_3['longitudinal']['printed_f'] + KEARNS_TABLE_3['transverse']['printed_f']:.4f}")
print()
print("A perfect fibre with all basal poles perpendicular to the axis would give")
print("(0.5, 0.5, 0.0). This rod is within a few percent of that, and its triad closes")
print("to about 3 percent -- which for 1965, with intensities integrated by planimeter")
print("and curves read by eye, is a good measurement. Note what the closure test cannot")
print("do: correcting the bad cell moves the sum by 0.002, well inside its own scatter,")
print("so it was the row arithmetic that exposed the slip, not the sum rule.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), sharey=False)
width = 8.0
for ax, (name, block) in zip(axes, KEARNS_TABLE_3.items(), strict=True):
    _, volume, contribution = kearns_columns(block["I"])
    ax.bar(phi_mid - 1.9, volume, width / 2, label=r"$V_{\Delta\phi}$  (volume)")
    ax.bar(
        phi_mid + 1.9,
        contribution,
        width / 2,
        label=r"$V_{\Delta\phi}\cos^{2}\bar\phi$  (contribution to $f$)",
    )
    ax.set_title(f"{name} section: $f$ = {contribution.sum():.3f}")
    ax.set_xlabel(r"tilt $\phi$ from the reference direction (degrees)")
    ax.set_xticks(phi_mid)
    ax.legend(fontsize=8)
axes[0].set_ylabel("fraction")
fig.suptitle("Kearns 1965 Table 3: where the volume is, and what it contributes", y=1.02)
fig.tight_layout()

The two panels are the whole argument in one picture. Both blocks put most of their
*volume* at high tilt — that is the $\sin\phi$ factor, and it is true of almost every
texture. What separates them is where that volume sits relative to $\cos^{2}\phi$: the
longitudinal block keeps a substantial share below $40^\circ$, where the property weight is
still large, while the transverse block has everything beyond $70^\circ$, where the weight
has essentially vanished. Hence $0.49$ against $0.05$ from columns that look, at a glance,
not so different.

## 5. The pole-figure route, and what truncation costs

Baron *et al.* (1990) define the Kearns coefficients as the pole-figure integral, which
is the tensor written out in polar coordinates. `kearns_from_pole_figure` evaluates it,
taking the integration weights from the figure's own `sampling` attribute: a pole *cloud*
carries its weights in its intensities, while a *sampled density* on a tilt raster needs
solid-angle weights, because a raster is not a uniform sampling of the sphere.

First the calibration — a pole figure computed from a known texture must give that
texture's $f$ back.

In [ ]:
bimodal = TEXTURES["bimodal split basal (+/-30 deg to TD)"]
truth = kearns_from_orientations(bimodal, pole=basal)
from_figure = kearns_from_pole_figure(PoleFigure.from_orientations(bimodal, basal))

print("from orientations:", np.round(np.asarray(truth.values), 6))
print("from pole figure :", np.round(np.asarray(from_figure.values), 6))
print("agree to:", float(np.max(np.abs(np.asarray(truth.values) - np.asarray(from_figure.values)))))

Now the systematic error that dominates real measurements. Reflection geometry defocuses
beyond about $75$–$80^\circ$ of tilt, so a measured figure stops short of the equator.
Since the missing cap is precisely where $\cos^2\phi \approx 0$, discarding it removes
the poles that pull $f$ down, and $f$ along the section normal comes out **too high**.

The experiment below measures the size of that bias directly: build a raster pole figure
from an analytic density, then integrate it over progressively smaller tilt ranges.

In [ ]:
from pytex.core.sphere import spherical_angles_to_directions


def raster_figure(density, *, step_deg=5.0, max_tilt_deg=90.0):
    polar = np.arange(0.0, max_tilt_deg + 1e-9, step_deg)
    azimuth = np.arange(0.0, 360.0, step_deg)
    polar_grid, azimuth_grid = np.meshgrid(polar, azimuth, indexing="ij")
    directions = spherical_angles_to_directions(polar_grid.ravel(), azimuth_grid.ravel())
    return PoleFigure(
        pole=basal,
        sample_directions=directions,
        intensities=density(directions),
        specimen_frame=specimen,
        sampling="sampled_density",
    )


def watson_density(kappa):
    return lambda v: np.exp(kappa * v[:, 2] ** 2) + 0.2


complete = kearns_from_pole_figure(raster_figure(watson_density(4.0))).value("ND")
print(f"complete figure (0-90 deg): f_ND = {complete:.4f}\n")
print(f"{'max tilt':>9} {'coverage':>9} {'f_ND':>8} {'error':>9}")
truncations = []
for max_tilt in (85.0, 80.0, 75.0, 70.0, 65.0, 60.0):
    report = kearns_from_pole_figure(raster_figure(watson_density(4.0), max_tilt_deg=max_tilt))
    value = report.value("ND")
    truncations.append((max_tilt, value))
    coverage = report.diagnostics["measured_solid_angle_fraction"]
    print(f"{max_tilt:8.0f}d {coverage:9.3f} {value:8.4f} {value - complete:+9.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
tilts, values = np.asarray(truncations).T
ax.plot(tilts, values, "o-", label="truncated measurement")
ax.axhline(complete, ls="--", color="0.4", label="complete figure")
ax.set_xlabel("maximum measured tilt (degrees)")
ax.set_ylabel(r"$f_{\rm ND}$")
ax.invert_xaxis()
ax.legend()
ax.set_title("Truncating a pole figure biases $f$ upward")
fig.tight_layout()

The bias is one-sided and grows fast. At the conventional $75^\circ$ limit it is already
several percent, and it is a *bias*, so a longer count time will not touch it. The
report says so itself rather than leaving the reader to notice.

In [ ]:
print(kearns_from_pole_figure(raster_figure(watson_density(4.0), max_tilt_deg=75.0)).describe())

## 6. The ODF route, and what the kernel costs

Reconstruct an ODF and take $\mathbf{A}$ from it. Its advantage over the pole-figure
route is that it does not need a strong $(0002)$ peak in the measured section:
alternative reflections plus the inversion supply the basal information. That is why it
stays usable in an ND–TD section of strongly basal-textured zirconium, where the
$(0002)$ intensity is negligible and the pole-figure route is dividing by noise.

But a kernel-density ODF is a *smoothed* estimate, and the smoothing pulls $\mathbf{A}$
toward isotropy. The size of that bias has a closed form. If a pole is smeared by a
rotation of angle $\omega$ about a uniform axis, Rodrigues' formula plus the moments
$\mathbb{E}[(\mathbf{a}\cdot\mathbf{c})^2]=1/3$, $\mathbb{E}[\dots^4]=1/5$ give
$\rho = \langle\cos^2\beta\rangle$ for the kernel, and then

$$\mathbf{A}_{\text{density}} = \tfrac{1}{3}\mathbf{I}
   + \beta\left(\mathbf{A}_{\text{support}} - \tfrac{1}{3}\mathbf{I}\right),
   \qquad \beta = \frac{3\rho-1}{2}.$$

Every departure from $1/3$ is scaled by $\beta$. Nothing here is fitted.

In [ ]:
print(f"{'halfwidth':>10} {'rho':>8} {'beta':>8}   effect on f = 0.70")
for halfwidth in (5.0, 10.0, 15.0, 20.0, 25.0, 30.0):
    rho = kernel_axis_shrinkage(KernelSpec(halfwidth_deg=halfwidth))
    beta = (3.0 * rho - 1.0) / 2.0
    smoothed = KEARNS_ISOTROPIC_VALUE + beta * (0.70 - KEARNS_ISOTROPIC_VALUE)
    print(f"{halfwidth:9.0f}d {rho:8.4f} {beta:8.4f}   0.70 reads as {smoothed:.4f}")

In [ ]:
fibre = basal_fibre([0.0, 0.0, 1.0], spread_deg=12.0, count=6000, seed=41)
exact = kearns_from_orientations(fibre, pole=basal).value("ND")

print(f"exact (from the orientations themselves): f_ND = {exact:.4f}\n")
print(f"{'halfwidth':>10} {'as a density':>13} {'as a support':>13}")
for halfwidth in (5.0, 10.0, 20.0, 30.0):
    odf = ODF.from_orientations(fibre, kernel=KernelSpec(halfwidth_deg=halfwidth))
    density = kearns_from_odf(odf, pole=basal).value("ND")
    support = kearns_from_odf(odf, pole=basal, deconvolve_kernel=True).value("ND")
    print(f"{halfwidth:9.0f}d {density:13.4f} {support:13.4f}")

Which column is right depends on where the ODF came from, which is why it is a parameter
and not a hidden choice:

- For an ODF **fitted by pole-figure inversion**, the weights were chosen so that the
  *smoothed* density matches the measurement. The smoothed density is the model of the
  material, so the density reading is correct — `deconvolve_kernel=False`, the default.
- For an ODF **built from measured orientations**, the support *is* the data and the
  kernel is only estimation blur. The support reading is correct —
  `deconvolve_kernel=True`, and it recovers the exact value above.

Either way: **report the halfwidth alongside any $f$ taken from an ODF.** At a
$20^\circ$ halfwidth the two readings differ by more than most of the differences
texture work argues about.

## 7. The diffractogram route

Kearns' own method, and the only one that needs no texture goniometer. In a symmetric
$\theta$–$2\theta$ scan only planes parallel to the surface diffract, so each peak's
integrated intensity — compared with the same peak from a random powder — is the pole
density of that $(hkil)$ along the section normal. The crystals contributing to it have
their basal pole at a fixed angle $\phi_{hkil}$ to that normal, so each reflection gives
one point of $I(\phi)$. The set of them is an inverse pole figure of the section normal,
which is why the literature also calls this the IPF route.

PyTex computes $\phi_{hkil}$ from the phase's own reciprocal metric rather than from a
transcribed table, so it follows the lattice parameters in use.

In [ ]:
from pytex.core.miller import plane_hkl_to_hkil_array
from pytex.texture.kearns import (
    DiffractogramReflection,
    basal_tilt_angle_deg,
    basal_tilt_profile,
    kearns_from_diffractogram,
)

# Kearns tabulated these by hand for c/a = 1.59 (his Table 2).
kearns_table_2 = {
    (0, 0, 0, 2): 0.0,
    (1, 0, -1, 5): 20.2,
    (1, 0, -1, 4): 24.7,
    (1, 0, -1, 3): 31.5,
    (1, 0, -1, 2): 42.5,
    (2, 0, -2, 3): 50.7,
    (1, 1, -2, 2): 57.8,
    (1, 0, -1, 1): 61.4,
    (2, 0, -2, 1): 74.8,
    (2, 1, -3, 1): 78.4,
    (1, 0, -1, 0): 90.0,
    (1, 1, -2, 0): 90.0,
}
print(f"{'(hkil)':>14} {'computed':>10} {'Kearns':>8}")
for quadruple, tabulated in kearns_table_2.items():
    plane = CrystalPlane.from_miller_bravais(quadruple, phase=zirconium)
    print(f"{str(quadruple):>14} {basal_tilt_angle_deg(plane):9.1f}d {tabulated:7.1f}d")

Note how the reflections cluster: nothing at all between $0^\circ$ and $20^\circ$, and
three of them piled up at $90^\circ$. The profile has to be interpolated across that gap,
and that interpolation — not the quadrature — is the route's dominant error. Here is the
whole chain validated end to end: simulate what a symmetric scan would record from a
known texture, then recover $f$ from those intensities alone.

In [ ]:
ALPHA_ZR_REFLECTIONS = (
    (0, 0, 0, 2), (1, 0, -1, 5), (1, 0, -1, 4), (1, 0, -1, 3), (2, 0, -2, 5),
    (1, 0, -1, 2), (2, 0, -2, 3), (1, 1, -2, 2), (1, 0, -1, 1), (2, 1, -3, 2),
    (2, 0, -2, 1), (2, 1, -3, 1), (1, 0, -1, 0),
)


def simulated_scan(orientations, *, axis=(0.0, 0.0, 1.0)):
    """The peak intensities a symmetric scan on this section would record."""
    poles = orientations.map_crystal_directions(basal.normal)
    vectors = np.asarray(getattr(poles, "values", poles), dtype=float)
    cosine = np.abs(vectors @ np.asarray(axis, dtype=float))
    edges = np.linspace(0.0, 90.0, 91)
    counts, _ = np.histogram(np.degrees(np.arccos(np.clip(cosine, 0.0, 1.0))), bins=edges)
    band = np.cos(np.deg2rad(edges[:-1])) - np.cos(np.deg2rad(edges[1:]))
    density = (counts / counts.sum()) / band
    midpoints = 0.5 * (edges[:-1] + edges[1:])
    return [
        DiffractogramReflection(
            plane=CrystalPlane.from_miller_bravais(q, phase=zirconium),
            intensity=float(
                np.interp(
                    basal_tilt_angle_deg(CrystalPlane.from_miller_bravais(q, phase=zirconium)),
                    midpoints,
                    density,
                )
            ),
            random_intensity=1.0,
        )
        for q in ALPHA_ZR_REFLECTIONS
    ]


print(f"{'texture':<34} {'exact f_ND':>11} {'from peaks':>11} {'error':>8}")
for label, spread in (("sharp fibre (10 deg)", 10.0), ("moderate (20 deg)", 20.0),
                      ("diffuse (35 deg)", 35.0), ("near random (70 deg)", 70.0)):
    orientations = basal_fibre([0.0, 0.0, 1.0], spread_deg=spread, count=60000, seed=51)
    exact = kearns_from_orientations(orientations, pole=basal).value("ND")
    recovered = kearns_from_diffractogram(
        simulated_scan(orientations), specimen_frame=specimen
    ).value("ND")
    print(f"{label:<34} {exact:11.4f} {recovered:11.4f} {recovered - exact:+8.4f}")

Better than its reputation: the interpolation error stays around a percent of $f$ across
the whole range of texture sharpness. What the route cannot do is give a triad from one
specimen — each section is a separate measurement, on a separately cut coupon.

### The reference intensities, and what normalizing them does not buy

Raw peak areas are not pole densities: $(10ar{1}1)$ and $(20ar{2}0)$ differ by a
factor of twenty in a *random* powder, from structure factor and multiplicity alone. So
every reflection needs a reference intensity $I_0$. It can be measured on a powder
standard, or calculated from the structure — only ratios enter $f$, so arbitrary units
are fine.

The classical alternative is the Harris texture coefficient, which rescales $I/I_0$ to a
mean of one over the measured reflections. Kearns tested that assumption against a real
standard and found the mean ran from 1.02 to 1.56, averaging 1.23, so it costs about
23 percent on absolute pole densities. On $f$ it costs **exactly nothing**, and the
reason is worth being explicit about: $f$ is a ratio of two integrals over the same
profile, so any common scale factor cancels identically.

In [ ]:
orientations = basal_fibre([0.0, 0.0, 1.0], spread_deg=20.0, count=60000, seed=51)
reflections = simulated_scan(orientations)
exact = kearns_from_orientations(orientations, pole=basal).value("ND")

with_standard = kearns_from_diffractogram(reflections, specimen_frame=specimen)
harris = kearns_from_diffractogram(reflections, specimen_frame=specimen, normalization="harris")

print(f"exact                        f_ND = {exact:.6f}")
print(f"reference intensities as given f_ND = {with_standard.value('ND'):.6f}")
print(f"Harris texture coefficients    f_ND = {harris.value('ND'):.6f}")
print(f"difference between the two          = {abs(harris.value('ND') - with_standard.value('ND')):.2e}")
print()
print("And with no reference intensities at all, the route refuses rather than")
print("quietly treating peak areas as densities:")
try:
    kearns_from_diffractogram(
        [
            DiffractogramReflection(plane=r.plane, intensity=r.intensity)
            for r in reflections
        ],
        specimen_frame=specimen,
    )
except ValueError as error:
    print(" ", str(error)[:110], "...")

## 8. Real data

Everything so far had a known answer. This section does not, and that is the point: the
checks that passed on simulated textures are the instruments for diagnosing real
measurements.

The data are Panalytical MRD scans of three zirconium products — a rolled Zircaloy-2
plate measured on three principal sections, a water-quenched Zr-2.5Nb, and a PHWR clad
tube — comprising four incomplete pole figures per section plus a $2\theta$ scan. They
are **not** part of this repository: they are several megabytes of instrument output, and
the repository holds sources and canonical assets only. Point `PYTEX_KEARNS_DATA` at a
copy, or place it at the default path below, and the rest of this section runs. Without
it, the cells say so and the notebook completes anyway.

In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path(
    os.environ.get(
        "PYTEX_KEARNS_DATA", "kearns_parameter_data_references/reference_exp_data"
    )
)
HAVE_DATA = DATA_ROOT.is_dir()

if HAVE_DATA:
    print(f"reference data found at {DATA_ROOT}")
else:
    print(f"reference data not found at {DATA_ROOT}.")
    print("Section 8 will report what it would have computed and continue.")

### 8.1 One basal pole figure per section

Each XRDML file is a $\psi$–$\varphi$ raster: tilt $0$ to $85^\circ$ in $5^\circ$ steps,
azimuth in $5^\circ$ steps, 72 points per ring. The adapter reads it; attaching the pole
and the specimen frame is a separate, explicit step, because a diffractometer file does
not know what it measured.

The specimen frame is labelled `(u, v, n)` rather than `(RD, TD, ND)` deliberately: `n`
is the section normal, and which physical direction that is depends on how the operator
cut and mounted the coupon — a convention the file does not record.

In [ ]:
from pytex.adapters.xrdml import read_xrdml_pole_figure

section_frame = ReferenceFrame(
    "section", FrameDomain.SPECIMEN, ("u", "v", "n"), Handedness.RIGHT
)

SPECIMENS = {
    "Zry-2 plate, section RD": "Zr2RolingTDND/RD/0 0 2.xrdml",
    "Zry-2 plate, section LT": "Zr2RolingTDND/LT/002.xrdml",
    "Zry-2 plate, section ST": "Zr2RolingTDND/ST/002.xrdml",
    "Zr-2.5Nb, water quenched": "Zr-2.5Nb-water quenched 840\u00f8/002.xrdml",
    "PHWR clad tube, final stage": "PHWR_CladTube_FinalStage/002.xrdml",
}

measured = {}
if HAVE_DATA:
    for label, relative in SPECIMENS.items():
        path = DATA_ROOT / relative
        if not path.exists():
            print(f"  missing: {relative}")
            continue
        scan = read_xrdml_pole_figure(path)
        figure = scan.to_pole_figure(basal, specimen_frame=section_frame)
        measured[label] = (scan, figure, kearns_from_pole_figure(figure))

    print(f"{'specimen':<30} {'f_u':>7} {'f_v':>7} {'f_n':>7} {'max tilt':>9}")
    for label, (scan, figure, report) in measured.items():
        print(
            f"{label:<30} {report.value('u'):7.4f} {report.value('v'):7.4f} "
            f"{report.value('n'):7.4f} {report.diagnostics['max_polar_deg']:8.0f}d"
        )
else:
    print("skipped: no reference data")

In [ ]:
if measured:
    fig, axes = plt.subplots(
        1, len(measured), figsize=(3.6 * len(measured), 4.0), subplot_kw={"aspect": "equal"}
    )
    axes = np.atleast_1d(axes)
    for ax, (label, (scan, figure, report)) in zip(axes, measured.items(), strict=True):
        plot_pole_figure(figure, ax=ax, title=f"{label}\n$f_n$ = {report.value('n'):.3f}")
    fig.suptitle("Measured (0002) pole figures", y=1.03)
    fig.tight_layout()
else:
    print("skipped: no reference data")

### 8.2 The check that fails

The three Zircaloy-2 files are three principal sections of one plate, so their three
*section-normal* values are the Kearns parameters along three mutually perpendicular
specimen directions. They must sum to 1.

In [ ]:
SECTION_KEYS = ["Zry-2 plate, section RD", "Zry-2 plate, section LT", "Zry-2 plate, section ST"]

if all(key in measured for key in SECTION_KEYS):
    normals = np.array([measured[key][2].value("n") for key in SECTION_KEYS])
    print(f"{'section':<10} {'f along its normal':>20}")
    for key, value in zip(SECTION_KEYS, normals, strict=True):
        print(f"{key[-2:]:<10} {value:20.4f}")
    print(f"\n  sum = {normals.sum():.4f}   (must be exactly 1)")
    print(f"  excess = {normals.sum() - 1.0:+.4f}, i.e. {100 * (normals.sum() - 1):.0f} percent")
    print("\n  after normalising to the sum rule:")
    for key, value in zip(SECTION_KEYS, normals / normals.sum(), strict=True):
        print(f"    f_{key[-2:]} = {value:.3f}")
else:
    print("skipped: no reference data")

A sum of roughly 1.4 is a large, one-sided error, and both of its causes are things this
notebook has already measured on simulated data:

1. **Truncation at $85^\circ$** (section 5). Every figure discards the outer cap, which is
   where $\cos^2$ of the angle to its own normal is smallest. Each $f_n$ is therefore
   biased *up*.
2. **Defocusing.** In reflection geometry the irradiated area grows and the diffracted
   intensity falls with tilt, again suppressing the outer rings and again biasing $f_n$
   up. Correcting it properly needs a random-powder standard measured under identical
   conditions; this dataset does not include one, which is itself the lesson —
   **measure the standard, or you cannot correct the figure.**

Both push the same way, which is why the excess is one-sided rather than scattered. The
industrial mitigation is exactly the normalization above: keep the section-normal value
from each section, where the well-measured low-tilt data dominate, and scale the triad to
sum to 1. Mani Krishna *et al.* (2011) reach the same recipe from the other end, by
comparing against EBSD.

Note also what the *within-figure* triad cannot tell you: a single pole figure determines
the whole tensor, so its own three values sum to 1 automatically. The sum rule is only a
test when the three numbers come from three independent measurements.

### 8.3 The ODF route on the same data

Four incomplete pole figures per section, inverted to an ODF, then resolved. Because the
inversion is driven by the same truncated and defocused intensities, it should inherit
the same bias — and confirming that is more useful than hoping otherwise.

In [ ]:
POLE_FILES = {
    "RD": [
        ("Zr2RolingTDND/RD/0 0 2.xrdml", (0, 0, 0, 2)),
        ("Zr2RolingTDND/RD/0 1 1.xrdml", (0, 1, -1, 1)),
        ("Zr2RolingTDND/RD/112.xrdml", (0, 1, -1, 2)),
        ("Zr2RolingTDND/RD/013.xrdml", (0, 1, -1, 3)),
    ],
    "LT": [
        ("Zr2RolingTDND/LT/002.xrdml", (0, 0, 0, 2)),
        ("Zr2RolingTDND/LT/011.xrdml", (0, 1, -1, 1)),
        ("Zr2RolingTDND/LT/012.xrdml", (0, 1, -1, 2)),
        ("Zr2RolingTDND/LT/013.xrdml", (0, 1, -1, 3)),
    ],
    "ST": [
        ("Zr2RolingTDND/ST/002.xrdml", (0, 0, 0, 2)),
        ("Zr2RolingTDND/ST/011.xrdml", (0, 1, -1, 1)),
        ("Zr2RolingTDND/ST/012.xrdml", (0, 1, -1, 2)),
        ("Zr2RolingTDND/ST/013.xrdml", (0, 1, -1, 3)),
    ],
}

odf_normals = {}
if HAVE_DATA:
    dictionary = OrientationSet.from_matrices(
        SciRot.random(1500, random_state=2).as_matrix(),
        crystal_frame=crystal,
        specimen_frame=section_frame,
        phase=zirconium,
        symmetry=symmetry,
    )
    print(f"{'section':<8} {'PF route':>10} {'ODF route':>11} {'residual':>10}")
    for section, entries in POLE_FILES.items():
        paths = [DATA_ROOT / relative for relative, _ in entries]
        if not all(path.exists() for path in paths):
            print(f"  missing files for section {section}")
            continue
        figures = [
            read_xrdml_pole_figure(path).to_pole_figure(
                CrystalPlane.from_miller_bravais(quadruple, phase=zirconium),
                specimen_frame=section_frame,
                intensity_normalization="mrd",
            )
            for path, (_, quadruple) in zip(paths, entries, strict=True)
        ]
        inversion = ODF.invert_pole_figures(
            figures,
            orientation_dictionary=dictionary,
            kernel=KernelSpec(halfwidth_deg=12.0),
            regularization=1e-3,
            max_iterations=200,
        )
        from_odf = kearns_from_odf(
            inversion.odf, pole=basal, deconvolve_kernel=True
        ).value("n")
        from_pf = kearns_from_pole_figure(figures[0]).value("n")
        odf_normals[section] = from_odf
        print(
            f"{section:<8} {from_pf:10.4f} {from_odf:11.4f} "
            f"{inversion.relative_residual_norm:10.3f}"
        )
    if len(odf_normals) == 3:
        total = sum(odf_normals.values())
        print(f"\n  ODF-route section-normal sum = {total:.4f}")
else:
    print("skipped: no reference data")

As expected, the two routes track each other closely and their sums are equally far from
1. Inverting to an ODF does not repair a systematic error in the intensities that feed
it; it inherits it, and adds a regularization choice on top. The ODF route's real
advantage is elsewhere — it survives a section where the $(0002)$ peak is too weak to
integrate, because the other three reflections still constrain the basal distribution.

### 8.4 The diffractogram route on the same plate

The $2\theta$ scans of the same three sections make the third route available on real
data. Two things have to be built first: the random-powder intensities, which PyTex
computes from the structure rather than taking from a table; and integrated peak areas,
which come from a local linear background under a window around each calculated $2\theta$.

In [ ]:
from pytex.app.phases import builtin_phase
from pytex.diffraction.xrd import generate_powder_reflections

powder_phase = builtin_phase("zr_hcp").to_phase()
powder = sorted(
    generate_powder_reflections(powder_phase, two_theta_range_deg=(25.0, 119.0), max_index=6),
    key=lambda reflection: reflection.two_theta_deg,
)
strongest = max(reflection.intensity for reflection in powder)

print(f"{'(hkil)':>14} {'2theta':>9} {'I_calc':>8} {'phi to c':>9}")
for reflection in powder[:10]:
    hkl = tuple(int(v) for v in reflection.miller_indices)
    hkil = tuple(int(v) for v in plane_hkl_to_hkil_array(np.asarray(hkl))[0])
    plane = CrystalPlane.from_miller_bravais(hkil, phase=zirconium)
    print(
        f"{str(hkil):>14} {reflection.two_theta_deg:8.2f}d "
        f"{100 * reflection.intensity / strongest:7.1f} "
        f"{basal_tilt_angle_deg(plane):8.1f}d"
    )

In [ ]:
import csv

OMEGA_DEG = 15.0  # the omega the scans were recorded at; see the discussion below

SCAN_FILES = {
    "RD": "Zr2RolingTDND/RD/2theta-rolling.csv",
    "LT": "Zr2RolingTDND/LT/2theta-longtrans.csv",
    "ST": "Zr2RolingTDND/ST/2theta-shorttrans.csv",
}


def read_scan(path):
    rows = list(csv.reader(path.open(encoding="latin-1")))
    start = next(i for i, row in enumerate(rows) if row and row[0].strip() == "[Scan points]")
    angle, counts = [], []
    for row in rows[start + 2 :]:
        if len(row) < 2 or not row[0].strip():
            break
        try:
            angle.append(float(row[0]))
            counts.append(float(row[1]))
        except ValueError:
            break
    return np.array(angle), np.array(counts)


def integrated_area(angle, counts, centre, half_window=1.2, background_span=0.6):
    """Net peak area over a locally fitted linear background."""
    inside = np.abs(angle - centre) <= half_window
    flank = (np.abs(angle - centre) > half_window) & (
        np.abs(angle - centre) <= half_window + background_span
    )
    if inside.sum() < 5 or flank.sum() < 4:
        return None
    slope, offset = np.polyfit(angle[flank], counts[flank], 1)
    net = counts[inside] - (slope * angle[inside] + offset)
    return float(np.trapezoid(np.clip(net, 0.0, None), angle[inside]))


def reflections_from_scan(path):
    angle, counts = read_scan(path)
    entries = []
    for reflection in powder:
        area = integrated_area(angle, counts, reflection.two_theta_deg)
        if area is None or area <= 0.0:
            continue
        hkl = tuple(int(v) for v in reflection.miller_indices)
        hkil = tuple(int(v) for v in plane_hkl_to_hkil_array(np.asarray(hkl))[0])
        entries.append(
            DiffractogramReflection(
                plane=CrystalPlane.from_miller_bravais(hkil, phase=zirconium),
                intensity=area,
                random_intensity=reflection.intensity,
                # The scans are fixed-omega, so the diffraction vector sits at
                # theta - omega from the surface normal. Recording it lets the
                # report say how far the assumption has been stretched.
                specimen_tilt_deg=0.5 * reflection.two_theta_deg - OMEGA_DEG,
            )
        )
    return entries


scan_normals = {}
if HAVE_DATA:
    print(f"{'section':<8} {'peaks':>6} {'f along normal':>15} {'tilt spread':>12}")
    for section, relative in SCAN_FILES.items():
        path = DATA_ROOT / relative
        if not path.exists():
            print(f"  missing: {relative}")
            continue
        entries = reflections_from_scan(path)
        report = kearns_from_diffractogram(entries, specimen_frame=section_frame)
        scan_normals[section] = report.value("n")
        print(
            f"{section:<8} {len(entries):6d} {report.value('n'):15.4f} "
            f"{report.diagnostics['diffraction_vector_tilt_spread_deg']:11.1f}d"
        )
    if len(scan_normals) == 3:
        total = sum(scan_normals.values())
        print(f"\n  section-normal sum = {total:.4f}")
        print("  after normalising:",
              {k: round(v / total, 3) for k, v in scan_normals.items()})
else:
    print("skipped: no reference data")

In [ ]:
if scan_normals and len(measured) >= 3:
    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    labels = list(SCAN_FILES)
    pf_values = np.array([measured[f"Zry-2 plate, section {s}"][2].value("n") for s in labels])
    scan_values = np.array([scan_normals[s] for s in labels])
    x = np.arange(len(labels))
    ax.bar(x - 0.2, pf_values / pf_values.sum(), 0.4, label="pole-figure route")
    ax.bar(x + 0.2, scan_values / scan_values.sum(), 0.4, label=r"$\theta$-$2\theta$ route")
    ax.axhline(KEARNS_ISOTROPIC_VALUE, ls="--", color="0.4", label="random, 1/3")
    ax.set_xticks(x, labels)
    ax.set_xlabel("section")
    ax.set_ylabel("$f$ along the section normal (normalised triad)")
    ax.legend()
    ax.set_title("Two independent routes on the same Zircaloy-2 plate")
    fig.tight_layout()
else:
    print("skipped: no reference data")

Two independent techniques, on different specimens cut from the same plate, agreeing on
the ordering and to within a few hundredths after normalization. That is about the
agreement the interlaboratory round-robin of Baron *et al.* (1990) found between five
laboratories using the same technique — a $4$–$10\%$ spread on the large coefficients and
$40\%$ on the small one — so it is a reasonable place to stop expecting better.

```{admonition} A geometry trap worth knowing
:class: warning
The `diffraction-vector tilt spread` reported above is not zero. These scans have
`scanAxis="2Theta"` with a single fixed `Omega` of $15^{\circ}$ — a **detector scan**, not
a coupled $\theta$–$2\theta$ scan. The diffraction vector therefore sits at
$\theta - \omega$ from the surface normal and swings from $2.5^{\circ}$ at the $(0002)$
peak to $45^{\circ}$ at the top of the range, so different reflections probe different
specimen directions. Kearns' derivation assumes a symmetric scan, where the spread is
zero.

The same corpus contains genuinely coupled scans, recorded as `scanAxis="Gonio"` with
*both* axes ranged, so the distinction is visible in the file rather than a matter of
recollection. `DiffractogramReflection` carries `specimen_tilt_deg` for exactly this
reason: the condition is reported instead of silently degrading the answer.
```

## 9. Choosing a route

| Route | Use it when | Its dominant error |
| --- | --- | --- |
| Discrete orientations (EBSD) | The microstructure indexes well — recrystallized material. Best consistency across sections. | Statistics, and orientation-correlated indexing failure in deformed material. |
| Pole figure | A basal pole figure exists and the section carries real $(0002)$ intensity. | Truncation and defocusing, both one-sided and both biasing $f$ up along the section normal. |
| ODF | The section's $(0002)$ peak is too weak to integrate; other reflections still constrain the basal distribution. | Inherits the pole figures' bias, and adds the kernel shrinkage and the inversion's regularization. |
| $\theta$–$2\theta$ diffractogram | No goniometer; or as an independent check. | Interpolation across the reflection gaps, the random standard, and needing three separate specimens. |

Whichever route: **quote the sum over the triad.** It is 1 by identity, it costs nothing
to compute, and it is the only diagnostic that needs no reference specimen. Everything
this notebook found in real data, it found through that number.

### See also

- [The Kearns Parameter And Basal-Pole Texture](../../theory/kearns_parameter_and_basal_pole_texture.md)
  — the derivations behind every formula used here.
- [Pole-Figure Arithmetic And The m.r.d. Scale](../../theory/pole_figure_arithmetic_and_mrd.md)
  — the raster quadrature the pole-figure route depends on.
- [Worked examples: the Kearns parameter](../../examples/generated/kearns-parameter.md)
  — the identities above, computed and checked on every documentation build.
- Tutorial 25, *Pole Figure Arithmetic*, and Tutorial 06, *Texture, ODF and pole-figure
  inversion*, for the objects this notebook consumes.